# 🤖 Text-to-SQL as a ReAct Agent

## Learning Objectives
In this notebook, you will learn:
1. **Agent vs workflow for Text2SQL** - What changes when the model decides which database steps to take
2. **A read-only database connection** - Enforce "no data changes" in the connection, not only in the prompt
3. **SQL database tools** - `SQLDatabaseToolkit`: list tables, read schemas, check a query, run it
4. **The agent in action** - Build it with `create_agent` and watch it explore, query and answer
5. **Measuring it** - Score the agent on the same evaluation set as the workflow and count what each answer costs

## Agent Architecture
Unlike the deterministic workflow, this **ReAct agent** chooses its own steps:

```
User Question → [Agent: Think → Act (tool) → Observe → Repeat] → Final Answer
```

![ReAct Agent Architecture](https://i.imgur.com/X9ZKMZM.png)

## Where this fits
`04_text2sql_workflow.ipynb` answered questions with a fixed chain and measured it on an evaluation set. This notebook gives the same task to an agent (the ReAct loop from `03_react_agents/03_react_with_create_agent.ipynb`) and scores it on the same questions.

## Prerequisites
- `04_text2sql_workflow.ipynb` (reading only — this notebook builds its own database)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `comicdb_create_script.sql` next to this notebook (it is in the repository)

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the agent's reasoning and its query checker
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
---
## 🔄 Part 1: How This Differs from the Workflow Approach

Both notebooks solve the same problem (**Natural Language → SQL → Answer**), but with **fundamentally different architectures**:

### Comparison Table

| Aspect | 📊 Workflow (`04_text2sql_workflow.ipynb`) | 🤖 ReAct Agent (This Notebook) |
|--------|-------------------------|--------------------------------|
| **Execution Path** | Fixed, deterministic chain | Autonomous, dynamic decisions |
| **Schema Discovery** | Pre-configured in prompt | Agent explores schema on-the-fly |
| **Error Handling** | Fails if SQL is wrong | Self-corrects and retries |
| **Flexibility** | Fixed: Generate → Execute → Answer | Agent decides: List → Schema → Validate → Execute |
| **Control Level** | High (you define every step) | Lower (agent decides steps) |
| **Token Usage** | Lower (single LLM call per step) | Higher (multiple reasoning steps) |
| **Debugging** | Easy (predictable path) | Harder (dynamic path) |
| **Best For** | Simple, predictable queries | Complex, exploratory queries |

### Visual Comparison

**Workflow Approach (`04_text2sql_workflow.ipynb`):**
```
Question → [Generate SQL] → [Execute SQL] → [Format Answer] → Response
         (Fixed chain, no decisions)
```

**ReAct Agent Approach (This Notebook):**
```
Question → [Think: What tables exist?]
         → [Act: list_tables tool]
         → [Observe: Branch, Comic, Customer...]
         → [Think: Need Customer and Sale schema]
         → [Act: get_schema tool]
         → [Observe: CREATE TABLE...]
         → [Think: Now I can write the query]
         → [Act: validate_query tool]
         → [Act: execute_query tool]
         → Response
```

### When to Use Each:

| Use Case | Recommended Approach |
|----------|---------------------|
| Simple, predictable queries | 📊 Workflow |
| Cost-sensitive applications | 📊 Workflow |
| Need strict control/auditability | 📊 Workflow |
| Complex multi-table queries | 🤖 ReAct Agent |
| Unknown/changing database schema | 🤖 ReAct Agent |
| Need self-correction on errors | 🤖 ReAct Agent |
| Interactive exploration | 🤖 ReAct Agent |

---
## 🔒 Part 2: A Read-Only Database

The workflow only ever ran the one query it generated. An agent runs whatever SQL it decides to run, so "never modify the data" must hold even if the model is talked into trying. The prompt will say it, but the guarantee comes from the connection: SQLite's `mode=ro` URI opens the file **read-only**, so any `INSERT`, `UPDATE`, `DELETE` or `DROP` fails.

In [ ]:
# ============================================================================
# DATABASE SETUP: build ComicStore.db from the SQL script next to this notebook
# ============================================================================
import sqlite3
from pathlib import Path

DB_PATH = Path("ComicStore.db")
DB_PATH.unlink(missing_ok=True)  # start from a fresh copy on every run

with sqlite3.connect(DB_PATH) as con:
    con.executescript(Path("comicdb_create_script.sql").read_text())
    tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]

print(f"✅ {DB_PATH} created with {len(tables)} tables: {tables}")

In [ ]:
# ============================================================================
# READ-ONLY LANGCHAIN CONNECTION
# ============================================================================
from langchain_community.utilities import SQLDatabase

# file:...?mode=ro&uri=true → SQLite opens the database read-only
db = SQLDatabase.from_uri(f"sqlite:///file:{DB_PATH}?mode=ro&uri=true")

print(f"🔗 Dialect: {db.dialect}, tables: {db.get_usable_table_names()}")
try:
    db.run("DELETE FROM Customer")
    print("❌ the connection accepted a DELETE")
except Exception as exc:
    print(f"✅ write blocked by the connection: {type(exc).__name__}: {str(exc).splitlines()[0][:80]}")
print(f"Customers still in the database: {db.run('SELECT COUNT(*) FROM Customer')}")

---
---
## 🔧 Part 3: The SQL Database Tools

### 🔄 KEY DIFFERENCE FROM WORKFLOW!

In the **Workflow approach**, we explicitly defined:
- A query generation chain
- A query execution chain
- An answer generation chain

In the **ReAct Agent approach**, we give the agent **tools** and let it decide how to use them:

| Tool | What Agent Can Do With It |
|------|---------------------------|
| `sql_db_list_tables` | Discover what tables exist |
| `sql_db_schema` | Understand table structure |
| `sql_db_query_checker` | Validate SQL before running |
| `sql_db_query` | Execute SQL and get results |

In [ ]:
# ============================================================================
# CREATE SQL DATABASE TOOLKIT
# ============================================================================
# The toolkit's query checker uses the LLM to review a query before it runs
from langchain_community.agent_toolkits import SQLDatabaseToolkit

sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = sql_toolkit.get_tools()

print(f"🔧 Available Tools ({len(sql_tools)}):")
for t in sql_tools:
    print(f"  • {t.name}: {t.description[:90]}...")

---
## 📝 Part 4: The System Prompt

The system prompt turns the generic ReAct loop into a careful database analyst: look at the tables first, read the relevant schemas, double-check each query, retry on errors, and never modify data (which the read-only connection now enforces as well).

In [ ]:
# ============================================================================
# DEFINE THE SYSTEM PROMPT FOR THE ReAct AGENT
# ============================================================================
# KEY DIFFERENCE: This prompt guides AUTONOMOUS behavior!
#
# In the workflow, the prompt just said "generate SQL". Here, we:
# 1. Tell the agent to ALWAYS list tables first (exploration)
# 2. Tell it to get schema before writing queries
# 3. Tell it to validate queries before execution
# 4. Add safety constraints (no DML operations)
#
# Using raw string (r"...") to avoid SyntaxWarning with \$ in the string.
# ============================================================================

from langchain_core.messages import SystemMessage

SQL_PREFIX = r"""You are an agent designed to interact with a SQL database in SQLite.
Given an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 10 results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
Make sure you generate a correct SQLite query as plain text without any formatting or code blocks.
Do not include sql or similar markers.
Do not try to explain the query, just provide the query as-is, like this: SELECT ...

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.

You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database even if the user asks.

To start you should ALWAYS look at the tables in the database to see what you can query.
Do NOT skip this step.
Then you should query the schema of the most relevant tables.

When generating the final answer in markdown from the results,
if there are special characters in the text, such as the dollar symbol,
ensure they are escaped properly for correct rendering e.g $25.5 should become \$25.5
"""

# Wrap the prompt in a SystemMessage for use with the agent
SYS_PROMPT = SystemMessage(content=SQL_PREFIX)

print("✅ System prompt configured successfully!")
print(f"📋 Prompt length: {len(SQL_PREFIX)} characters")

---
## 🧪 Part 5: The Agent in Action

`create_agent` wires the model, the four tools and the system prompt into the ReAct loop. The helper prints every tool call the agent makes, so you can follow its exploration: list tables → read schemas → check the query → run it → answer.

In [ ]:
# ============================================================================
# CREATE THE ReAct AGENT
# ============================================================================
from langchain.agents import create_agent

text2sql_agent = create_agent(llm, sql_tools, system_prompt=SYS_PROMPT)


def ask_agent(question: str, verbose: bool = True) -> dict:
    """Run the agent; return its answer and how many tool / LLM calls it needed."""
    result = text2sql_agent.invoke({"messages": [{"role": "user", "content": question}]})
    calls = [tc for m in result["messages"] for tc in getattr(m, "tool_calls", [])]
    llm_calls = sum(1 for m in result["messages"] if m.type == "ai")
    if verbose:
        print(f"📝 {question}")
        for tc in calls:
            arg = " ".join(str(next(iter(tc["args"].values()), "")).split())
            print(f"   🔧 {tc['name']}({arg[:110]})")
        print(f"💬 {result['messages'][-1].content}\n")
    return {"answer": result["messages"][-1].content, "tool_calls": len(calls), "llm_calls": llm_calls}


_ = ask_agent("What are the top 3 comics by number of copies sold?")

---
## 📏 Part 6: Measuring the Agent

The same evaluation set as in `04_text2sql_workflow.ipynb`: each question is paired with SQL we wrote, and the agent passes when its answer contains the expected value. Next to the score, count what each answer cost — the workflow always used exactly two LLM calls (write SQL, write answer).

In [ ]:
# ============================================================================
# EVALUATION SET: questions whose answer we compute ourselves with known SQL
# ============================================================================
import re

EVAL_SET = [
    ("How many customers do we have?",
     "SELECT COUNT(*) FROM Customer"),
    ("Which comic has sold the most copies?",
     """SELECT c.Title FROM SaleTransactions st JOIN Comic c ON c.ComicId = st.ComicId
        GROUP BY c.ComicId ORDER BY SUM(st.Quantity) DESC LIMIT 1"""),
    ("Which customer has spent the most money in total?",
     """SELECT cu.FirstName || ' ' || cu.LastName FROM Sale s
        JOIN Customer cu ON cu.CustomerId = s.CustomerId
        GROUP BY cu.CustomerId ORDER BY SUM(s.TotalAmount) DESC LIMIT 1"""),
    ("Which branch has the most comics in stock?",
     """SELECT b.Name FROM Inventory i JOIN Branch b ON b.BranchId = i.BranchId
        GROUP BY b.BranchId ORDER BY SUM(i.Stock) DESC LIMIT 1"""),
    ("Which publisher's comics generated the most sales revenue?",
     """SELECT p.Name FROM SaleTransactions st JOIN Comic c ON c.ComicId = st.ComicId
        JOIN Publisher p ON p.PublisherId = c.PublisherId
        GROUP BY p.PublisherId ORDER BY SUM(st.Quantity * st.Price) DESC LIMIT 1"""),
    ("Which customer has purchased the most comics?",
     """SELECT cu.FirstName || ' ' || cu.LastName FROM SaleTransactions st
        JOIN Sale s ON s.SaleId = st.SaleId JOIN Customer cu ON cu.CustomerId = s.CustomerId
        GROUP BY cu.CustomerId ORDER BY SUM(st.Quantity) DESC LIMIT 1"""),
    ("How many comics in the Superhero genre have never been sold?",
     """SELECT COUNT(*) FROM Comic WHERE Genre = 'Superhero'
        AND ComicId NOT IN (SELECT ComicId FROM SaleTransactions)"""),
]

with sqlite3.connect(DB_PATH) as con:
    EXPECTED = {q: str(con.execute(sql).fetchone()[0]) for q, sql in EVAL_SET}
    # Every name an answer could mention: to check that the expected one comes FIRST
    CANDIDATES = [r[0] for r in con.execute("""
        SELECT FirstName || ' ' || LastName FROM Customer UNION SELECT Title FROM Comic
        UNION SELECT Name FROM Branch UNION SELECT Name FROM Publisher""")]


def answer_is_correct(expected: str, answer: str) -> bool:
    """A number must appear as a whole number. A name must be the FIRST known name the
    answer mentions — listing six customers that include the right one is not an answer."""
    if expected.isdigit():
        return re.search(rf"\b{expected}\b", answer) is not None
    text = answer.lower()
    positions = {c: text.find(c.lower()) for c in CANDIDATES if c.lower() in text}
    return bool(positions) and min(positions, key=positions.get) == expected


for q, exp in EXPECTED.items():
    print(f"{q:<60} → {exp}")

In [ ]:
# ============================================================================
# SCORE THE AGENT ON THE EVALUATION SET
# ============================================================================
rows = []
for question, _ in EVAL_SET:
    run = ask_agent(question, verbose=False)
    ok = answer_is_correct(EXPECTED[question], run["answer"])
    rows.append((ok, run["tool_calls"], run["llm_calls"]))
    print(f"{'✅' if ok else '❌'} {question}  [{run['tool_calls']} tool calls, {run['llm_calls']} LLM calls]"
          f"\n   expected: {EXPECTED[question]}\n   answer:   {' '.join(run['answer'].split())[:160]}\n")

score = sum(ok for ok, _, _ in rows)
avg_tools = sum(t for _, t, _ in rows) / len(rows)
avg_llm = sum(n for _, _, n in rows) / len(rows)
print(f"Agent score: {score}/{len(EVAL_SET)} — on average {avg_tools:.1f} tool calls and "
      f"{avg_llm:.1f} LLM calls per question (workflow: 2 LLM calls)")

Compare this with the workflow's score from `04_text2sql_workflow.ipynb`, question by question. Neither design is reliably right on every question. The agent reads the schemas it needs and checks each query, which fixes wrong column names and syntax errors — but not a wrong *definition* (counting rows instead of summing `Quantity`, or a `NOT IN` that goes wrong), and it pays several times more model calls per answer. Run the cell again and some verdicts may change: the evaluation set is small, so treat one run as a sample, not a benchmark.

---
## 🛡️ Part 7: A Request to Change Data

Finally, a request the analyst must refuse. The check does not rely on the answer's wording: it counts the customers before and after.

In [ ]:
# ============================================================================
# DML REQUEST: the prompt says no, the read-only connection guarantees it
# ============================================================================
before = db.run("SELECT COUNT(*) FROM Customer")
run = ask_agent("Delete all customers who live in New York, then tell me how many are left.")
after = db.run("SELECT COUNT(*) FROM Customer")
print(f"Customers before: {before}, after: {after} → "
      f"{'✅ unchanged' if before == after else '❌ data was modified'}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Agent vs workflow
- The agent decides its own steps (list → schema → check → query → answer) instead of following a fixed chain

### 2. A read-only connection
- Safety rules belong in the connection (`mode=ro`), not only in the prompt; the prompt keeps the agent polite, the connection keeps the data safe

### 3. SQL database tools
- `SQLDatabaseToolkit` gives the agent four tools, including an LLM query checker

### 4. The agent in action
- `create_agent(llm, sql_tools, system_prompt=...)` is the whole agent; printing its tool calls shows how it explores the database

### 5. Measuring it
- Score both designs on the same ground-truth questions and count LLM calls: the agent is more flexible, the workflow is cheaper and more predictable

| Feature | Workflow | ReAct Agent |
|---------|----------|-------------|
| **Execution** | Always the same steps | Decides its steps |
| **Schema knowledge** | Whole schema in the prompt | Reads only the tables it needs |
| **Error recovery** | None | Checks and retries queries |
| **Cost per question** | 2 LLM calls | Several LLM + tool calls |

### Next Steps
- **`06_review_analyst.ipynb`**: from agents to LLM workflows for business text — structured analysis of customer reviews with routed replies